# AAL Sales Analysis - Q4 2020
## Objective
Analyze Australian Apparel Ltd (AAL) sales data for Q4 2020 to identify high-revenue states and develop intervention strategies for underperforming regions.

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy

# Styling and display options
sns.set_style("darkgrid")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

## 1. Data Wrangling
### 1.1 Load and Validate

In [7]:
import os

# Path to CSV (relative to notebook location)
filepath = os.path.join('../data', 'AusApparalSales4thQrt2020.csv')
if not os.path.exists(filepath):
    raise FileNotFoundError(f'File not found: {filepath}')

df = pd.read_csv(filepath)

print('Shape:', df.shape)
print('Info:')
df.info()

df.head()

if df.empty:
    raise ValueError('Loaded DataFrame is empty — check the source CSV or path.')

Shape: (7560, 6)
Info:
<class 'pandas.DataFrame'>
RangeIndex: 7560 entries, 0 to 7559
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Date    7560 non-null   str  
 1   Time    7560 non-null   str  
 2   State   7560 non-null   str  
 3   Group   7560 non-null   str  
 4   Unit    7560 non-null   int64
 5   Sales   7560 non-null   int64
dtypes: int64(2), str(4)
memory usage: 354.5 KB


In [11]:
# Inspect missing / present values
na_counts = df.isna().sum()
print('Missing values per column:')
print(na_counts)

# Show a quick boolean mask (first rows) to inspect not-null entries
print()
print('Not-null entries (first 5 rows):')
df.notna().head()

Missing values per column:
Date     0
Time     0
State    0
Group    0
Unit     0
Sales    0
dtype: int64

Not-null entries (first 5 rows):


,Date,Time,State,Group,Unit,Sales
0,True,True,True,True,True,True
1,True,True,True,True,True,True
2,True,True,True,True,True,True
3,True,True,True,True,True,True
4,True,True,True,True,True,True


### 1.2 Data Cleaning and Aggregation

Aggregation strategy: create multiple dimensional views to support exploration and visualization.
- Keep `df_daily` as the canonical daily-grain table (Date, Time, State, Group, Unit, Sales).
- Build focused rollups for time and dimensional analysis: week/time, state/group, group/state, daily, weekly, monthly, quarterly.
- Each rollup will be reset-indexed and surfaced with shape and a sample for quick inspection.

In [19]:
# Build canonical daily-grain and multiple aggregation views for analysis
required_cols = ['Date', 'Time', 'State', 'Group', 'Unit', 'Sales']
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise KeyError(f'Missing required columns for aggregation: {missing}')

# df_daily: daily-grain (one row per Date, Time, State, Group)
df_daily = df.groupby(['Date', 'Time', 'State', 'Group'], as_index=False).agg({'Unit': 'sum', 'Sales': 'sum'})
df_daily.reset_index(drop=True, inplace=True)
# Ensure Date is datetime for period-based rollups
df_daily['Date'] = pd.to_datetime(df_daily['Date'])
try:
    df_daily['Week'] = df_daily['Date'].dt.isocalendar().week
except Exception:
    df_daily['Week'] = df_daily['Date'].dt.week
df_daily['Month'] = df_daily['Date'].dt.month
df_daily['Quarter'] = df_daily['Date'].dt.quarter

# 1) Week-Time rollup
agg_time_week = df_daily.groupby(['Week', 'Time'], as_index=False).agg({'Unit': 'sum', 'Sales': 'sum'})
print(f'agg_time_week: {agg_time_week.shape}')
print(agg_time_week.head())

# 2) State-Group rollup
agg_state_group = df_daily.groupby(['State', 'Group'], as_index=False).agg({'Unit': 'sum', 'Sales': 'sum'})
print(f'agg_state_group: {agg_state_group.shape}')
print(agg_state_group.head())

# 3) Group-State rollup
agg_group_state = df_daily.groupby(['Group', 'State'], as_index=False).agg({'Unit': 'sum', 'Sales': 'sum'})
print(f'agg_group_state: {agg_group_state.shape}')
print(agg_group_state.head())

# 4) Daily totals
agg_daily = df_daily.groupby(['Date'], as_index=False).agg({'Unit': 'sum', 'Sales': 'sum'})
print(f'agg_daily: {agg_daily.shape}')
print(agg_daily.head())

# 5) Weekly totals
agg_weekly = df_daily.groupby(['Week'], as_index=False).agg({'Unit': 'sum', 'Sales': 'sum'})
print(f'agg_weekly: {agg_weekly.shape}')
print(agg_weekly.head())

# 6) Monthly totals
agg_monthly = df_daily.groupby(['Month'], as_index=False).agg({'Unit': 'sum', 'Sales': 'sum'})
print(f'agg_monthly: {agg_monthly.shape}')
print(agg_monthly.head())

# 7) Quarterly totals
agg_quarterly = df_daily.groupby(['Quarter'], as_index=False).agg({'Unit': 'sum', 'Sales': 'sum'})
print(f'agg_quarterly: {agg_quarterly.shape}')
print(agg_quarterly.head())

agg_time_week: (42, 4)
   Week        Time  Unit    Sales
0    40   Afternoon  1926  4815000
1    40     Evening  2019  5047500
2    40     Morning  2073  5182500
3    41   Afternoon  3527  8817500
4    41     Evening  3634  9085000
agg_state_group: (28, 4)
  State     Group  Unit     Sales
0   NSW      Kids  7435  18587500
1   NSW       Men  7609  19022500
2   NSW   Seniors  7275  18187500
3   NSW     Women  7669  19172500
4    NT      Kids  2280   5700000
agg_group_state: (28, 4)
   Group State  Unit     Sales
0   Kids   NSW  7435  18587500
1   Kids    NT  2280   5700000
2   Kids   QLD  3404   8510000
3   Kids    SA  5806  14515000
4   Kids   TAS  2310   5775000
agg_daily: (90, 3)
        Date  Unit    Sales
0 2020-10-01  1488  3720000
1 2020-10-02  1486  3715000
2 2020-10-03  1556  3890000
3 2020-10-04  1488  3720000
4 2020-10-05  1545  3862500
agg_weekly: (14, 3)
   Week   Unit     Sales
0    40   6018  15045000
1    41  10801  27002500
2    42  10656  26640000
3    43  10726  2681

In [18]:
print("Raw df shape:", df.shape)
print("\nUnique combinations of (Date, Time, State, Group):")
unique_combos = df.groupby(['Date', 'Time', 'State', 'Group']).size()
print("Count of unique combos:", len(unique_combos))
print("\nFirst 10 unique combos:")
print(unique_combos.head(10))

Raw df shape: (7560, 6)

Unique combinations of (Date, Time, State, Group):
Count of unique combos: 7560

First 10 unique combos:
Date        Time       State  Group  
1-Dec-2020  Afternoon  NSW    Kids       1
                              Men        1
                              Seniors    1
                              Women      1
                       NT     Kids       1
                              Men        1
                              Seniors    1
                              Women      1
                       QLD    Kids       1
                              Men        1
dtype: int64


In [17]:
# Derive time periods from Date
df_daily['Date'] = pd.to_datetime(df_daily['Date'])
try:
    df_daily['Week'] = df_daily['Date'].dt.isocalendar().week
except Exception:
    df_daily['Week'] = df_daily['Date'].dt.week
df_daily['Month'] = df_daily['Date'].dt.month
df_daily['Quarter'] = df_daily['Date'].dt.quarter
print(df_daily.head())
df_daily.shape

        Date        Time State     Group  Unit  Sales  Week  Month  Quarter
0 2020-12-01   Afternoon   NSW      Kids    31  77500    49     12        4
1 2020-12-01   Afternoon   NSW       Men    36  90000    49     12        4
2 2020-12-01   Afternoon   NSW   Seniors    30  75000    49     12        4
3 2020-12-01   Afternoon   NSW     Women    38  95000    49     12        4
4 2020-12-01   Afternoon    NT      Kids     9  22500    49     12        4


(7560, 9)

### 1.3 Normalization

In [ ]:
# Placeholder for normalization logic — implement later
pass

## 2. Data Analysis
### 2.1 Descriptive Statistics

In [ ]:
# Placeholder for descriptive statistics (mean, median, mode, std)
# To be implemented later
pass

### 2.2 Highest and Lowest Sales

In [ ]:
# Placeholder for identifying highest/lowest by State and Group
pass

### 2.3 Time-Based Aggregations

In [ ]:
# Placeholder for weekly, monthly, quarterly rollups
pass

## 3. Data Visualization

### 3.1 State-wise Sales by Group

In [ ]:
# Placeholder for state-wise subplots
pass

### 3.2 Group-wise Sales by State

In [ ]:
# Placeholder for group-wise subplots
pass

### 3.3 Time-of-Day Analysis (Peak/Off-Peak)

In [ ]:
# Placeholder for time-of-day subplots
pass

## 4. Insights and Recommendations

[Placeholder for business insights]